# TrOCR — Transformer-Based OCR with CER and WER Evaluation

In [ ]:
!pip install -q transformers torch pillow datasets sentencepiece

In [ ]:
import reimport torchfrom PIL import Image, ImageDraw, ImageFontfrom transformers import TrOCRProcessor, VisionEncoderDecoderModel

In [ ]:
DATASET_PATH = ""MODEL_NAME = "microsoft/trocr-base-printed"device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
def create_sample_image(text, width=600, height=150, font_size=32):    image = Image.new("RGB", (width, height), color=(255, 255, 255))    draw = ImageDraw.Draw(image)    try:        font = ImageFont.truetype("DejaVuSans.ttf", font_size)    except IOError:        font = ImageFont.load_default()    draw.text((15, height // 2 - font_size // 2), text, fill=(0, 0, 0), font=font)    return image

In [ ]:
def load_evaluation_samples(dataset_path):    if dataset_path:        from datasets import load_dataset        dataset = load_dataset(dataset_path, split="test")        images = [sample["image"].convert("RGB") for sample in dataset]        references = [sample["text"] for sample in dataset]        return images, references    references = [        "The quick brown fox jumps over the lazy dog.",        "TrOCR combines a vision transformer with a text decoder.",        "Optical character recognition converts images into text.",        "Attention is all you need for sequence modeling.",    ]    images = [create_sample_image(text) for text in references]    return images, referencesimages, references = load_evaluation_samples(DATASET_PATH)images[0]

In [ ]:
processor = TrOCRProcessor.from_pretrained(MODEL_NAME)model = VisionEncoderDecoderModel.from_pretrained(MODEL_NAME).to(device)model.eval()

In [ ]:
def prepare_batch(images, processor):    pixel_values = processor(images=images, return_tensors="pt").pixel_values    return pixel_values

In [ ]:
def run_inference(model, processor, images, batch_size=4, max_length=64):    hypotheses = []    for start in range(0, len(images), batch_size):        batch_images = images[start:start + batch_size]        pixel_values = prepare_batch(batch_images, processor).to(device)        with torch.no_grad():            generated_ids = model.generate(pixel_values, max_length=max_length)        decoded = processor.batch_decode(generated_ids, skip_special_tokens=True)        hypotheses.extend(decoded)    return hypotheseshypotheses = run_inference(model, processor, images)for reference, hypothesis in zip(references, hypotheses):    print("ref:", reference)    print("hyp:", hypothesis)    print("-" * 40)

In [ ]:
def normalize_words(text, case_sensitive=False):    if not case_sensitive:        text = text.lower()    text = re.sub(r"[^a-z0-9' ]+", " ", text) if not case_sensitive else text    text = re.sub(r"\s+", " ", text).strip()    return text.split() if text else []def normalize_chars(text, case_sensitive=False):    if not case_sensitive:        text = text.lower()    text = re.sub(r"\s+", " ", text).strip()    return list(text)

In [ ]:
def edit_distance_ops(ref_units, hyp_units):    n, m = len(ref_units), len(hyp_units)    dp = [[0] * (m + 1) for _ in range(n + 1)]    for i in range(n + 1):        dp[i][0] = i    for j in range(m + 1):        dp[0][j] = j    for i in range(1, n + 1):        for j in range(1, m + 1):            if ref_units[i - 1] == hyp_units[j - 1]:                dp[i][j] = dp[i - 1][j - 1]            else:                substitution = dp[i - 1][j - 1] + 1                insertion = dp[i][j - 1] + 1                deletion = dp[i - 1][j] + 1                dp[i][j] = min(substitution, insertion, deletion)    i, j = n, m    substitutions = insertions = deletions = 0    while i > 0 or j > 0:        if i > 0 and j > 0 and ref_units[i - 1] == hyp_units[j - 1]:            i -= 1            j -= 1        elif i > 0 and j > 0 and dp[i][j] == dp[i - 1][j - 1] + 1:            substitutions += 1            i -= 1            j -= 1        elif j > 0 and dp[i][j] == dp[i][j - 1] + 1:            insertions += 1            j -= 1        else:            deletions += 1            i -= 1    return substitutions, deletions, insertions, dp[n][m]

In [ ]:
def cer(reference, hypothesis):    ref_chars = normalize_chars(reference)    hyp_chars = normalize_chars(hypothesis)    substitutions, deletions, insertions, _ = edit_distance_ops(ref_chars, hyp_chars)    n = len(ref_chars) if len(ref_chars) > 0 else 1    return (substitutions + deletions + insertions) / ndef wer(reference, hypothesis):    ref_words = normalize_words(reference)    hyp_words = normalize_words(hypothesis)    substitutions, deletions, insertions, _ = edit_distance_ops(ref_words, hyp_words)    n = len(ref_words) if len(ref_words) > 0 else 1    return (substitutions + deletions + insertions) / n

In [ ]:
def batch_evaluate(pairs):    total_char_s = total_char_d = total_char_i = total_char_n = 0    total_word_s = total_word_d = total_word_i = total_word_n = 0    per_example = []    for reference, hypothesis in pairs:        ref_chars = normalize_chars(reference)        hyp_chars = normalize_chars(hypothesis)        cs, cd, ci, _ = edit_distance_ops(ref_chars, hyp_chars)        char_n = len(ref_chars) if len(ref_chars) > 0 else 1        example_cer = (cs + cd + ci) / char_n        ref_words = normalize_words(reference)        hyp_words = normalize_words(hypothesis)        ws, wd, wi, _ = edit_distance_ops(ref_words, hyp_words)        word_n = len(ref_words) if len(ref_words) > 0 else 1        example_wer = (ws + wd + wi) / word_n        per_example.append({"cer": example_cer, "wer": example_wer})        total_char_s += cs        total_char_d += cd        total_char_i += ci        total_char_n += len(ref_chars)        total_word_s += ws        total_word_d += wd        total_word_i += wi        total_word_n += len(ref_words)    corpus_char_n = total_char_n if total_char_n > 0 else 1    corpus_word_n = total_word_n if total_word_n > 0 else 1    return {        "per_example": per_example,        "corpus_cer": (total_char_s + total_char_d + total_char_i) / corpus_char_n,        "corpus_wer": (total_word_s + total_word_d + total_word_i) / corpus_word_n,        "char_substitutions": total_char_s,        "char_deletions": total_char_d,        "char_insertions": total_char_i,        "word_substitutions": total_word_s,        "word_deletions": total_word_d,        "word_insertions": total_word_i,    }

In [ ]:
pairs = list(zip(references, hypotheses))results = batch_evaluate(pairs)for (reference, hypothesis), scores in zip(pairs, results["per_example"]):    print(f"ref: {reference}")    print(f"hyp: {hypothesis}")    print(f"cer: {scores['cer']:.3f}  wer: {scores['wer']:.3f}")    print("-" * 40)print(f"corpus cer: {results['corpus_cer']:.3f}")print(f"corpus wer: {results['corpus_wer']:.3f}")print(f"char substitutions: {results['char_substitutions']}  deletions: {results['char_deletions']}  insertions: {results['char_insertions']}")print(f"word substitutions: {results['word_substitutions']}  deletions: {results['word_deletions']}  insertions: {results['word_insertions']}")